[<img src="imagens/colab-badge.png" style="width:16%; vertical-align:middle;">](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap03/cap03_aluno.ipynb)
[<img src="imagens/github-badge.png" style="width:20%; vertical-align:middle;">](https://github.com/fzampirolli/pdi-vc)

In [ ]:
import os, importlib, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Baixar morph.py se necessário
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph
importlib.reload(morph)
from morph import mm

print("✅ Ambiente pronto")

In [ ]:
# Imagem de exemplo
url = "https://upload.wikimedia.org/wikipedia/en/7/7d/Lenna_%28test_image%29.png"
img_color = mm.read(url)
img = mm.gray(img_color)
print(f"Imagem carregada: {img.shape} | Tipo: {img.dtype}")

## 3.1 Convolução e Filtragem Espacial

A **convolução** é a operação fundamental das filtragens lineares. Para uma imagem $I$ e um kernel $K$ de tamanho $m \times n$, a convolução é definida como:

<a id="eq-03-convolucao"></a>
$$
(I * K)(x,y) = \sum_{i=-a}^{a} \sum_{j=-b}^{b} I(x+i, y+j) \cdot K(i,j) \tag{3.1}
$$


onde $(a,b)$ são os deslocamentos a partir do centro do kernel.

**Correlação** é semelhante, porém sem inversão do kernel (usada com mais frequência em PDI).

### 3.1.1 Filtros de Suavização

Os filtros de suavização reduzem ruído e detalhes de alta frequência.

In [ ]:
img_media3 = mm.filter_mean(img, 3)
img_media7 = mm.filter_mean(img, 7)
img_gauss = mm.filter_gaussian(img, 5, sigma=1.5)
img_mediana = mm.filter_median(img, 5)

mm.show([img, img_media3, img_media7, img_gauss, img_mediana],
        titles=["Original", "Média 3×3", "Média 7×7", "Gaussiano", "Mediana"],
        cols=5)

**Figura 3.1:** Comparação de filtros de suavização: Média 3x3, Média 7x7, Gaussiano e Mediana.


### 3.1.2 Filtros de Realce e Detecção de Bordas

Filtros de alta-passagem realçam bordas e detalhes.

In [ ]:
sobelx = mm.sobel_x(img)
sobely = mm.sobel_y(img)
sobel = mm.gradient_magnitude(img)
lap = mm.laplacian(img)

mm.show([img, sobelx, sobely, sobel, lap],
        titles=["Original", "Sobel X", "Sobel Y", "Magnitude Sobel", "Laplaciano"],
        cols=5)

**Figura 3.2:** Detecção de bordas com Sobel e Laplaciano.


## 3.2 Transformações Geométricas Avançadas

### 3.2.1 Transformação Afim Genérica

In [ ]:
M = mm.get_affine_matrix(angle=30, scale=1.2, shear=0.3)
img_afim = mm.warp_affine(img, M)

mm.show([img, img_afim], titles=["Original", "Afim (30°, 1.2x, shear 0.3)"])

**Figura 3.3:** Exemplo de transformação afim combinada (rotação + escala + shear).


### 3.2.2 Correção de Perspectiva (Homografia)

In [ ]:
url_sudoku = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/imagens/sudoku.jpg"
sudoku = mm.gray(mm.read(url_sudoku))

# Pontos de exemplo (ajuste conforme sua imagem)
pts_orig = np.float32([[50,50], [300,30], [350,350], [80,380]])
pts_dest = np.float32([[0,0], [300,0], [300,300], [0,300]])

img_corrigida = mm.warp_perspective(sudoku, pts_orig, pts_dest)

mm.show([sudoku, img_corrigida], titles=["Original", "Corrigida (Homografia)"])

**Figura 3.4:** Correção de perspectiva em imagem de Sudoku.


## 3.3 Processamento Morfológico

### 3.3.1 Operações Básicas: Erosão e Dilatação

In [ ]:
kernel = mm.structuring_element(5, shape='rectangle')

erosao = mm.erode(img, kernel)
dilatacao = mm.dilate(img, kernel)

mm.show([img, erosao, dilatacao],
        titles=["Original", "Erosão", "Dilatação"])

**Figura 3.5:** Erosão e Dilatação com kernel 5x5.


### 3.3.2 Abertura e Fechamento

In [ ]:
abertura = mm.opening(img, kernel)
fechamento = mm.closing(img, kernel)

mm.show([img, abertura, fechamento],
        titles=["Original", "Abertura", "Fechamento"])

**Figura 3.6:** Abertura (erosão + dilatação) e Fechamento (dilatação + erosão).


## 3.4 Resumo

Neste capítulo exploramos as principais **operações espaciais**:

- Convolução e filtragem espacial (suavização e realce)
- Transformações geométricas afins e homografias
- Morfologia Matemática: erosão, dilatação, abertura e fechamento

O Capítulo 4 abordará o **Processamento no Domínio da Frequência** (Transformada de Fourier).

## 3.5 🤖 Uso do NotebookLM como Tutor Complementar

> ### ❗ 🎓 Estude com o Tutor Inteligente
>
> [🚀 ACESSAR NOTEBOOKLM: CAPÍTULO 03](https://notebooklm.google.com/notebook/XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX)

## 3.6 Lista de Exercícios

1. **(15%)** Explique a diferença entre convolução e correlação. Por que em PDI usamos mais a correlação?

2. **(20%)** Aplique um filtro de média 5×5 e um Gaussiano equivalente na imagem Lena. Qual apresenta melhor preservação de bordas? Justifique.

3. **(20%)** Implemente manualmente (usando loops ou NumPy) um filtro Laplaciano 3×3 e compare com `mm.laplacian`.

4. **(15%)** Mostre que a abertura remove pequenos objetos claros enquanto o fechamento remove pequenos buracos escuros.

5. **(15%)** Corrija a perspectiva de uma imagem real (ex: cartão de visita ou página de livro) usando homografia.

6. **(15%)** Combine erosão/dilatação para remover ruído sal-pimenta de uma imagem binária.

## Referências do Capítulo


* Gonzalez e Woods (2018) — Processamento Digital de Imagens
* Szeliski (2022) — Computer Vision: Algorithms and Applications
* Bradski e Kaehler (2008) — Learning OpenCV

BRADSKI, Gary; KAEHLER, Adrian. **Learning OpenCV: Computer vision with the OpenCV library**. " O'Reilly Media, Inc.", 2008.

GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. New York, Pearson, 2018.

SZELISKI, Richard. **Computer Vision: Algorithms and Applications**. Cham, Switzerland, Springer, 2022.